## Sachai Check · Urdu audio deepfake detection
XLS-R 300M + layer-weighted SLS head, speaker-disjoint CSALT split. Every cell is safe to re-run: downloads, caching, training and evaluation resume or skip finished work.

In [ ]:
import json, time
from pathlib import Path
from dotenv import load_dotenv
load_dotenv()

import pandas as pd, torch
from IPython.display import Image, display
from sachai_audio.config import Config
from sachai_audio import data, train as T, evaluate as E, plots as P
from sachai_audio.infer import Detector
from sachai_audio.explain import explain
from sachai_audio.hub import push_to_hub

# Defaults fit the RTX A4000 (16 GB): batch 16 x accum 2, 4 workers. On an 8 GB GPU add batch_size=8, grad_accum=4.
cfg = Config(project_dir=".", run_name="xlsr300m-sls-v1")
FORCE_RETRAIN = False          # True only if you really want to start over

device = T.pick_device()
print(device, torch.cuda.get_device_name(0) if device.type == "cuda" else "", "| torch", torch.__version__)

### 1 · Data: download, 16 kHz cache, frozen speaker split

In [ ]:
mpath = cfg.root / "data" / "manifest.csv"
if not mpath.exists():
    data.download_dataset(cfg)                   # resumable, skips finished files
    manifest = data.preprocess_all(data.make_splits(data.build_manifest(cfg), cfg), cfg)   # resumable cache
manifest = pd.read_csv(mpath)
print(json.loads((cfg.root / "data" / "splits.json").read_text()))
manifest[manifest.keep].groupby(["split", "generator"]).size().unstack()

### 2 · Shortcut audit (can metadata alone predict the label?)

In [ ]:
audit_tbl, leak = data.audit(manifest)
display(audit_tbl.round(3))
leak

### 3 · Train (power cut or Ctrl+C? just run this cell again)

In [ ]:
summary = T.train(cfg, manifest, device, force_restart=FORCE_RETRAIN)
summary

### 4 · Under- or overfitting?

In [ ]:
display(Image(P.learning_curves(cfg.run_dir / "history.csv", cfg.report_dir)))
print(P.fit_diagnosis(cfg.run_dir / "history.csv"))

### 5 · Calibrate, set the decision band, export (dev speakers only)

In [ ]:
E.finalize(cfg, manifest, device)                # skips if export already matches the best weights
json.loads((cfg.export_dir / "policy.json").read_text())

### 6 · Held-out test speakers (frozen policy, run once)

In [ ]:
test = E.evaluate_test(cfg, manifest, device)
display(pd.read_csv(cfg.report_dir / "table_main.csv").round(4))
display(pd.read_csv(cfg.report_dir / "test_per_attack.csv").round(4))
display(pd.read_csv(cfg.report_dir / "test_per_speaker.csv").round(4))
print("95% CI (grouped bootstrap):", test["test"]["ci95"])
pol = E.load_policy(cfg.export_dir)
for f in (P.score_hist(cfg.report_dir / "test_scores.csv", pol, cfg.report_dir),
          P.roc_plot(cfg.report_dir / "test_scores.csv", cfg.report_dir),
          P.reliability(cfg.report_dir / "test_scores.csv", cfg.report_dir)):
    display(Image(f))

### 7 · Robustness: noise, reverb, phone band, MP3, Opus voice notes

In [ ]:
rob = E.robustness(cfg, manifest, device)        # each condition cached; resumes if interrupted
display(rob.round(4))
display(Image(P.robustness_plot(cfg.report_dir / "table_robustness.csv", cfg.report_dir)))

### 8 · Analysis for the paper: layer weights and acoustic descriptors

In [ ]:
display(Image(P.layer_weights_plot(cfg.export_dir, cfg.report_dir)))
E.descriptor_analysis(cfg, manifest).round(3)

### 9 · Product check: one clip through detector + LLM justifier + judge

In [ ]:
det = Detector(cfg.export_dir)                   # loads once, never retrains
row = manifest[(manifest.split == "test") & (manifest.label == 1) & manifest.keep].iloc[0]
res = det.analyze_file(cfg.raw_dir / row.path)
n_train_spk = len(json.loads((cfg.root / "data" / "splits.json").read_text())["train"])
ex = explain(res, n_train_spk)
print(row.path, "->", res["label_en"], res["spoof_score"], "| out of distribution:", res["ood"]["flag"])
for b in ex["bullets"]:
    print(" -", b)
print(ex["summary_ur"], "\n", ex["source"], ex.get("judge", ""))

t = time.time()
for _ in range(5):
    det.analyze_file(cfg.raw_dir / row.path, with_descriptors=False)
print(f"latency: {(time.time() - t) / 5 * 1000:.0f} ms per clip ({row.sec:.1f} s audio)")

### 10 · Push to Hugging Face (`HF_TOKEN` in `.env`)

In [ ]:
push_to_hub(cfg.export_dir)                      # private by default; HF_PRIVATE=false makes it public

### 11 · Serve the product
In a terminal, from this folder: `uvicorn api.main:app --host 127.0.0.1 --port 8000`, then open http://127.0.0.1:8000 and use the Audio Deepfake tab.